In [2]:
import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageSequence

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import classification_report, confusion_matrix

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [3]:
TRAIN_CSV = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\train_grouped.csv"
VAL_CSV   = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\val_grouped.csv"
TEST_CSV  = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\test_grouped.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)

train_df.head()

train: (4145, 24)
val  : (889, 24)
test : (889, 24)


,gif_id,gif_path,primary_emotion,primary_score,total_comparisons,pleasure_score,disgust_score,happiness_score,pride_score,excitement_score,...,satisfaction_score,guilt_score,contempt_score,shame_score,anger_score,amusement_score,contentment_score,relief_score,frame_path,emotion_group
0,cOHfCWdjBWg5G,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,sadness,42,184,1,16,4,4,2,...,2,24,7,26,10,1,2,3,processed_frames_single\train\cOHfCWdjBWg5G.jpg,negative_subdued
1,12wYFFcvnuAsRq,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,amusement,44,215,26,3,27,23,19,...,25,1,4,2,3,44,22,2,processed_frames_single\train\12wYFFcvnuAsRq.jpg,positive_energetic
2,ST9Iiel42U0Vi,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,fear,20,96,7,10,3,5,14,...,6,4,2,3,1,6,6,3,processed_frames_single\train\ST9Iiel42U0Vi.jpg,negative_intense
3,g2L4WvIB1JLfq,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,satisfaction,34,244,33,3,28,15,21,...,34,9,9,8,5,23,20,12,processed_frames_single\train\g2L4WvIB1JLfq.jpg,positive_calm
4,5WI25EWixXc5y,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,excitement,34,199,24,3,21,18,34,...,19,1,6,2,6,19,16,4,processed_frames_single\train\5WI25EWixXc5y.jpg,positive_energetic


In [4]:
# Groups are already in emotion_group column
label_col = "emotion_group"

all_labels = sorted(train_df[label_col].dropna().unique().tolist())
label2id = {l:i for i,l in enumerate(all_labels)}
id2label = {i:l for l,i in label2id.items()}

def add_label_id(df):
    df = df.copy()
    df["label_id"] = df[label_col].map(label2id).astype(int)
    return df

train_df = add_label_id(train_df)
val_df   = add_label_id(val_df)
test_df  = add_label_id(test_df)

def balance_table(df, name):
    counts = df[label_col].value_counts().reindex(all_labels, fill_value=0)
    perc = (counts / counts.sum() * 100).round(2)
    out = pd.DataFrame({"count": counts, "percent": perc})
    max_c = counts.max()
    min_c = counts[counts>0].min() if (counts>0).any() else 0
    ratio = round(float(max_c / min_c), 2) if min_c else None
    print(f"\n=== {name} ===")
    print(out)
    print("Total:", int(counts.sum()))
    print("Imbalance ratio (max/min):", ratio)

balance_table(train_df, "TRAIN")
balance_table(val_df, "VAL")
balance_table(test_df, "TEST")


=== TRAIN ===
                    count  percent
emotion_group                     
contempt              123     2.97
negative_intense     1023    24.68
negative_subdued      684    16.50
positive_calm         681    16.43
positive_energetic   1262    30.45
surprise              372     8.97
Total: 4145
Imbalance ratio (max/min): 10.26

=== VAL ===
                    count  percent
emotion_group                     
contempt               27     3.04
negative_intense      219    24.63
negative_subdued      146    16.42
positive_calm         147    16.54
positive_energetic    271    30.48
surprise               79     8.89
Total: 889
Imbalance ratio (max/min): 10.04

=== TEST ===
                    count  percent
emotion_group                     
contempt               26     2.92
negative_intense      219    24.63
negative_subdued      147    16.54
positive_calm         146    16.42
positive_energetic    271    30.48
surprise               80     9.00
Total: 889
Imbalance ratio (m

In [5]:
def imbalance_ratio(df):
    c = df["emotion_group"].value_counts()
    return float(c.max()/c.min())

print("Train imbalance ratio:", imbalance_ratio(train_df))

Train imbalance ratio: 10.260162601626016
